In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Estimación de uso: un minuto en un procesador Eagle (NOTA: Esto es solo una estimación. Su tiempo de ejecución puede variar.)*

## Antecedentes

Circuit-knitting es un término general que engloba varios métodos para particionar un circuito en múltiples subcircuitos más pequeños que involucran menos compuertas y/o qubits. Cada uno de los subcircuitos puede ejecutarse de forma independiente y el resultado final se obtiene mediante algún posprocesamiento clásico sobre el resultado de cada subcircuito. Esta técnica es accesible en el [complemento de Qiskit para corte de circuitos](https://qiskit.github.io/qiskit-addon-cutting/index.html), y se proporciona una explicación detallada de la técnica en la [documentación](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) junto con otro [material introductorio](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html).

Este cuaderno trata sobre un método llamado <b>corte de cables</b> donde el circuito se particiona a lo largo del cable [\[1\], \[2\]](#references). Tenga en cuenta que la partición es simple en circuitos clásicos ya que el resultado en el punto de partición puede determinarse de forma determinista, y es 0 o 1. Sin embargo, el estado del qubit en el punto del corte es, en general, un estado mixto. Por lo tanto, cada subcircuito necesita ser medido múltiples veces en diferentes bases (generalmente un conjunto tomográficamente completo de bases como la base de Pauli [\[3\], \[4\]](#references)) y correspondientemente preparado en su autoestado. La figura a continuación (<i>cortesía: Tesis de PhD, Ritajit Majumdar</i>) muestra un ejemplo de corte de cables para un estado GHZ de 4 qubits dividido en tres subcircuitos. Aquí $M_j$ denota un conjunto de bases (generalmente Pauli X, Y y Z) y $P_i$ denota un conjunto de autoestados (generalmente $|0\rangle$, $|1\rangle$, $|+\rangle$ y $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Dado que cada subcircuito tiene menos qubits y/o compuertas, se espera que sean menos susceptibles al ruido. Este cuaderno muestra un ejemplo donde este método puede utilizarse para suprimir eficazmente el ruido en el sistema.

## Requisitos

Antes de comenzar este tutorial, asegúrese de tener instalado lo siguiente:

- Qiskit SDK v2.0 o posterior, con soporte de [visualización](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 o posterior ( `pip install qiskit-ibm-runtime` )
- Complemento de Qiskit para corte de circuitos v0.9.0 o posterior (`pip install qiskit-addon-cutting`)

Consideraremos un circuito de Localización de Muchos Cuerpos (MBL, por sus siglas en inglés) para este cuaderno. El circuito MBL es un circuito eficiente en hardware y está parametrizado por dos parámetros $\theta$ y $\vec{\phi}$. Cuando $\theta$ se establece en $0$ y el estado inicial se prepara en $|0\rangle$ para todos los qubits, el valor esperado ideal de $\langle Z_i \rangle$ es $+1$ para cada sitio de qubit $i$ independientemente de los valores de $\vec{\phi}$. Puede consultar más detalles sobre los circuitos MBL en <a href="https://arxiv.org/abs/2307.07552">este artículo</a>.

## Configuración

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Parte I. Ejemplo a pequeña escala

### Paso 1: Transformar las entradas clásicas en un problema cuántico

Inicialmente construimos un circuito plantilla sin valores de parámetros específicos. También proporcionamos marcadores de posición, llamados `CutWire`, para anotar la posición de los cortes. Para el ejemplo a pequeña escala consideramos un circuito MBL de 10 qubits.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Ahora anotamos el circuito para el corte insertando las instrucciones **CutWire** apropiadas para crear dos cortes aproximadamente iguales. Establecemos `use_cut=True` en la función y permitimos que anote después de $\frac{n}{2}$ qubits, siendo $n$ el número de qubits en el circuito original.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### Paso 2: Optimizar el problema para la ejecución en hardware cuántico
A continuación, cortamos el circuito en dos subcircuitos más pequeños. Para este ejemplo, nos limitamos a solo 2 subcircuitos. Para ello, utilizamos el <a href="https://qiskit.github.io/qiskit-addon-cutting/">Complemento de Qiskit: Corte de Circuitos</a>.
#### Cortar el circuito en subcircuitos más pequeños
Cortar el cable en un punto aumenta el recuento de qubits en uno. Además del qubit original, ahora hay un qubit adicional como marcador de posición para el circuito después del corte. La siguiente imagen proporciona una representación:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

Este complemento utiliza la función `cut_wires` para tener en cuenta los qubits adicionales que surgen debido al corte.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### Crear y expandir los observables
Ahora construimos el observable $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$. Dado que el resultado ideal de $\langle Z_i \rangle$ para cada $i$ es $+1$, el resultado ideal de $M_z$ también es $+1$.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Sin embargo, tenga en cuenta que el número de qubits en el circuito ha aumentado después de insertar las operaciones virtuales `Move` de 2 qubits tras el corte. Por lo tanto, necesitamos expandir los observables también insertando identidades para ajustarlos al circuito actual.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Visualicemos los subcircuitos

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

Los observables también han sido particionados para ajustarse a los subcircuitos

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

Tenga en cuenta que cada subcircuito genera una serie de muestras. La reconstrucción tiene en cuenta el resultado de cada una de estas muestras. Cada una de estas muestras se denomina un `subexperiment`.
Extender el observable utilizando la operación `Move` requiere una estructura de datos `PauliList`. También podemos crear el observable $M_z$ en la estructura de datos más genérica `SparsePauliOp`, que será útil más adelante durante la reconstrucción de los subexperimentos.

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Veamos dos ejemplos donde los qubits cortados se miden en dos bases diferentes. Primero, se mide en la base Z normal, y luego se mide en la base X.

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### Transpilar cada subexperimento

Actualmente necesitamos transpilar nuestros circuitos antes de enviarlos para su ejecución. Por lo tanto, transpilaremos cada circuito en los subexperimentos primero.